In [12]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf

# spark = SparkSession.builder.appName("K8sClusterModeJob").getOrCreate()

# hadoop_conf = spark.hadoopConfiguration()
# hadoop_conf.set("fs.s3a.endpoint", "http://minio-storage:9000")
# hadoop_conf.set("fs.s3a.access.key","minioadmin")
# hadoop_conf.set("fs.s3a.secret.key","minioadmin")
# hadoop_conf.set("fs.s3a.path.style.access","true")
# hadoop_conf.set("fs.s3a.impl","org.apache.hadoop.fs.s3a.S3AFileSystem")

spark = SparkSession.builder \
    .appName("CheckResult") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio-storage:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

# df = spark.read.csv("s3a://test-bucket/2019-Nov.csv",header=True, inferSchema=True)
df = spark.read.csv("s3a://test-bucket/sample_2019.csv", header=True, inferSchema=True)
pid_code_map = df.filter(sf.col('category_code').isNotNull()) \
    .select('product_id', 'category_code') \
    .dropDuplicates(['product_id']) \
    .withColumnRenamed('category_code', 'code_by_pid')

pid_brand_map = df.filter(sf.col('brand').isNotNull()) \
    .select('product_id', 'brand') \
    .dropDuplicates(['product_id']) \
    .withColumnRenamed('brand', 'brand_by_pid')

cid_map = df.filter(sf.col('category_code').isNotNull()) \
    .select('category_id', 'category_code') \
    .dropDuplicates(['category_id']) \
    .withColumnRenamed('category_code', 'code_by_cid')

df_join_temp = df.join(sf.broadcast(pid_code_map), on='product_id', how='left') \
       .join(sf.broadcast(pid_brand_map), on='product_id', how='left') \
       .join(sf.broadcast(cid_map), on='category_id', how='left')

df_join = df_join_temp.withColumn('category_code',sf.coalesce(sf.col('category_code'),sf.col('code_by_cid'),sf.col('code_by_pid'))) \
                        .withColumn('brand',sf.coalesce(sf.col('brand'),sf.col('brand_by_pid'))) \
                        .drop('code_by_cid','code_by_pid','brand_by_pid')            


refined_df = df_join.filter(df_join['category_code'].isNotNull() & df_join['brand'].isNotNull())
refined_df = refined_df.drop('product_id','category_id','user_id','user_session')
refined_df = refined_df.withColumn("event_date", sf.to_date(sf.col("event_time")))

refined_df.write.repartition('category_code').mode('Overwrite').partitionBy('category_code').parquet("s3a://test-bucket/silver/ecommerce_refined")


In [13]:
print(df.count())
print(refined_df.count())
print(df.count() - refined_df.count())

100000
59442
40558


In [14]:
df.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



In [16]:
df_join.filter(sf.col('category_code').isNull() | sf.col('brand').isNull()).count()

40558

In [18]:
df_join.filter(sf.col('category_code').isNotNull() & sf.col('brand').isNotNull()).count()

59442

In [19]:
df.filter(sf.col('category_code').isNotNull() & sf.col('brand').isNotNull()).count()

59442

In [20]:
df_join.explain(True)

== Parsed Logical Plan ==
Project [category_id#896L, product_id#895, event_time#893, event_type#894, category_code#990, brand#1003, price#899, user_id#900, user_session#901]
+- Project [category_id#896L, product_id#895, event_time#893, event_type#894, category_code#990, coalesce(brand#898, brand_by_pid#919) AS brand#1003, price#899, user_id#900, user_session#901, code_by_pid#914, brand_by_pid#919, code_by_cid#924]
   +- Project [category_id#896L, product_id#895, event_time#893, event_type#894, coalesce(category_code#897, code_by_cid#924, code_by_pid#914) AS category_code#990, brand#898, price#899, user_id#900, user_session#901, code_by_pid#914, brand_by_pid#919, code_by_cid#924]
      +- Project [category_id#896L, product_id#895, event_time#893, event_type#894, category_code#897, brand#898, price#899, user_id#900, user_session#901, code_by_pid#914, brand_by_pid#919, code_by_cid#924]
         +- Join LeftOuter, (category_id#896L = category_id#971L)
            :- Project [product_id#895